# Kommentarer på vad vi ska göra

* Vi kan ha en champion för varje och säga att vi hitta genom att göra CV samt med grid search, man kan nämnn generellt vad man testat

* Komma på vilken metric är vettig att dra resultat av och ge till user

* Ha några plots, som train/val loss, confusion matrix och vad mer vi tycker är intressant

* Ha animering på video på cut video?? 

* Om tid finns kanske lägga till conv NN

* Göra allt för Kinect och media pipe data, ha olika champions då borde vara bäst?? Jämföra med varandra


# Sebastians generella instruktioner från moodle 

* Submit a single, self-contained, and fully runnable notebook - I do run and debug all of your notebooks. While I do inspect your repositories, you must not rely on me trying to find where the training code is, it needs to be in the notebook.

* Do not include the datasets or pre-rendered plots. Do not put Python in formatted markdown cells. Once I run the notebook, it computes all results. If your notebook relies on earlier results or produced data, you should include it though.

* Of course you can keep the reporting for all trained variants and champion models, etc. But the notebook needs to contain one runnable version of each variant so to speak.

* Submit just the notebook, not a Zip. Only a Zip if your notebook requires auxiliary files to run.

## Handle all inputs

In [ ]:
import torch.nn as nn
import torch
import mlflow
import random
import numpy as np
## lägg till allt som används

## Setup

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")      # Mac GPU (Apple Silicon)
elif torch.cuda.is_available():
    device = torch.device("cuda")     # Nvidia GPU
else:
    device = torch.device("cpu")

# make sure everything has same seeds
random_seed = 42
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("Squat_or_not")
mlflow.enable_system_metrics_logging()

## Load data 

* First load from chosen folder 
* Split into X and Y ignore frame NO feature
* Do normalisation with base in train data on all partitions




## Create the new Y value using the kinetic cut as reference

# **Feed forward network**


## Functions for the implementation

In [ ]:
# Function to load champion model/ best performing model
def load_champion_info(metadata_dir):
    path = os.path.join(metadata_dir, "champion_info.json")

    if not os.path.exists(path):
        return None

    try:
        with open(path, "r") as f:
            return json.load(f)
    except:
        return None

# Function for saving a model as our current champion model
def save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters):
    model_path = os.path.join(champion_dir, "champion_model.pt")
    info_path = os.path.join(metadata_dir, "champion_info.json")

    # Save model weights
    torch.save(model.state_dict(), model_path)

    # Save metadata
    info = {
        "model_name": model_name,
        "saved_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "f1": float(f1),
        "recall": float(recall),
        "precision": float(precision),
        "hyperparameters": hyperparameters
    }

    with open(info_path, "w") as f:
        json.dump(info, f, indent=2)

    print("New champion model saved!")


# Function that saves a model as champion uisng the save_champion_model function if either 1. We have no champion, 2. New model is better
def update_champion(metadata_dir, champion_dir, model, model_name, f1, recall, precision, hyperparameters):
    current = load_champion_info(metadata_dir)

    if current is None:
        print("No champion found --> saving first model")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)

    elif f1 > current["f1"]:
        print(f"New model is better (f1 {f1} > {current['f1']})")
        save_champion_model(champion_dir, metadata_dir, model, model_name, f1, recall, precision, hyperparameters)

    else:
        print(f"Model NOT better (f1 {f1} < {current['f1']})")


# Define initial weights and biases
def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.kaiming_uniform_(m.weight) # good for ReLU
        nn.init.zeros_(m.bias)


# Define Feedforward squat classifier class
class Squatclassifier(nn.Module):
    def __init__(self, input_size, hidden_layers: list, activation="relu", dropout=0.0):
        super().__init__()

        layers = []

        activations = {"relu": nn.ReLU(),
                       "tanh": nn.Tanh(),
                       "gelu": nn.GELU(),
                       "leaky_relu": nn.LeakyReLU()
                       }
        
        act = activations[activation]
        prev_size = input_size

        # Hidden layers
        for hidden_size in hidden_layers:
            layers.append(nn.Linear(prev_size, hidden_size))
            layers.append(act)

            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            
            prev_size = hidden_size

        # Output layer- 1 neuron
        layers.append(nn.Linear(prev_size, 1))

        self.network = nn.Sequential(*layers)
        self.network.apply(init_weights)
    
    def forward(self, X):
        return self.network(X)
        

# Initializes a squatclassifier model based on a configuration
def build_model(config):
    return Squatclassifier(
        input_size=config["input_size"],
        hidden_layers=config["layers"],
        activation=config["activation"],
        dropout=config["dropout"]
    ).to(device)


# Train the model
def train_one_model(model, config, x_train, y_train, x_val, y_val, loss_fn):
    optimizer = optim.Adam(model.parameters(), lr=config["lr"])
    epochs = config["epochs"]

    best_val_f1 = 0
    best_state = None
    patience = 10
    epochs_no_improve = 0

    # reshape targets
    y_train = y_train.view(-1, 1)
    y_val = y_val.view(-1, 1)

    # Training loop (full-batch)
    for epoch in range(epochs):

        model.train()
        optimizer.zero_grad()

        logits = model(x_train)
        loss = loss_fn(logits, y_train)

        loss.backward()
        optimizer.step()

        # Validation
        model.eval()

        with torch.no_grad():
            val_logits = model(x_val)
            probs = torch.sigmoid(val_logits)
            preds = (probs > 0.5).float()

            val_f1 = f1_score(
                y_val.cpu().numpy(),
                preds.cpu().numpy()
            )

        # Early stopping on f1-score
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = model.state_dict()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    return best_val_f1, best_state


# Cross validation function
def cross_validation(config, X, Y, k):

    kf = KFold(n_splits=k, shuffle=True, random_state=42)

    fold_scores = []
    fold_models = []

    # Train and evaluate on each fold
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        x_train = X[train_idx].to(device)
        y_train = Y[train_idx].to(device)

        x_val = X[val_idx].to(device)
        y_val = Y[val_idx].to(device)

        # Build model
        model = build_model(config)


        val_f1, best_state = train_one_model(
            model,
            config,
            x_train,
            y_train,
            x_val,
            y_val,
            # Binary Cross Entropy/Log Loss
            loss_fn=nn.BCEWithLogitsLoss()
        )

        fold_scores.append(val_f1)
        fold_models.append(best_state)


    avg_f1 = sum(fold_scores) / len(fold_scores)


    return {
        "cv_mean_f1": avg_f1,
        "fold_scores": fold_scores,
        "fold_models": fold_models
    }

## Setup

In [ ]:
# Gridsearch and cross validation where we decided upon the hyperparameter is in HugoProject/notebooks/assignment11_hugo.ipynb

# Access where best FNN classificator is stored
model_root = "binary_classificator_models/fnn_classificator"
metadata_path = os.path.join(model_root, "metadata", "champion_info.json")


champion_info = load_champion_info(metadata_path)
best_config = champion_info["hyperparameters"]
print("Champion config loaded")
print(best_config)


# Alternatively here is the config stored in local variable
# best_config = {"model_name": "final_model",
#                "saved_at": "2026-04-28 20:11:13",
#                "f1": 0.9239302694136292,
#                "recall": 0.938430583501006,
#                "precision": 0.9098712446351931,
#                "hyperparameters": {
#                 "layers": [
#                   256,
#                   128,
#                   32
#                 ],
#                 "lr": 0.005,
#                 "dropout": 0.1,
#                 "activation": "relu",
#                 "epochs": 200,
#                 "input_size": 39
#                 }
# }

## Re-train

In [ ]:
# Rebuild best model from cross validation grid search
final_model = build_model(best_config)

# Make split for the retraining s.t model isn't overfitted to the training data
x_tr, x_val, y_tr, y_val = train_test_split(x_train, y_train, test_size=0.1, random_state=42)

# Retrain best config from the cross validation gridsearch
_, best_state = train_one_model(final_model, best_config, x_tr, y_tr, x_val, y_val, loss_fn = nn.BCEWithLogitsLoss())


## Evaluate and results



In [ ]:
# Load weights from re-training
final_model.load_state_dict(best_state)

# Evaluate against test set
final_model.eval()
with torch.no_grad():
    logits = final_model(x_test.to(device))
    probs = torch.sigmoid(logits)
    preds = (probs > 0.5).float()

    f1 = f1_score(y_test.cpu(), preds.cpu())
    precision = precision_score(y_test.cpu(), preds.cpu())
    recall = recall_score(y_test.cpu(), preds.cpu())
    accuracy = accuracy_score(y_test.cpu(), preds.cpu())

    cm = confusion_matrix(y_test.cpu(), preds.cpu())

print("TEST RESULTS")
print(f"F1: {f1}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"Accuracy: {round(accuracy * 100, 4)}%")
print("\nConfusion Matrix for best performing FNN:")

# Plot Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Not Squat", "Squat"],
            yticklabels=["Not Squat", "Squat"])

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix for best performing FNN")
plt.show()

# Reccurant NN

## Setup

## Train

## Evaluate and results


# Cut video 

# Summary and future works